In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.regression import RandomForestRegressionModel

spark = SparkSession.builder \
    .appName("SCNE UI Preparation") \
    .master("local[*]") \
    .getOrCreate()

DATA_PATH = r"D:\Big Data Programming Project\Final Assignment\data\processed\spark\scne_model_features"
MODEL_PATH = r"D:\Big Data Programming Project\Final Assignment\models\random_forest_delay_model"

model_df = spark.read.parquet(DATA_PATH)
rf_model = RandomForestRegressionModel.load(MODEL_PATH)

print("Rows:", model_df.count())
print("Random Forest model loaded successfully")

Rows: 308885
Random Forest model loaded successfully


In [2]:
route_options = (
    model_df
    .select("published_line_name")
    .distinct()
    .orderBy("published_line_name")
    .toPandas()["published_line_name"]
    .tolist()
)

print("Total routes:", len(route_options))
print(route_options[:20])

Total routes: 62
['1', '10', '101', '11', '12', '13', '14', '16', '18', '2', '20', '22', '23', '3', '30', '31', '32', '32A', '34', '35']


In [3]:
from pyspark.ml.feature import StringIndexer

train_df = model_df.filter(
    model_df.service_date.isin("2025-12-26", "2025-12-27")
)

route_indexer = StringIndexer(
    inputCol="published_line_name",
    outputCol="route_index",
    handleInvalid="keep"
)

route_index_model = route_indexer.fit(train_df)

print("Route encoder prepared")
print("Known route labels:", len(route_index_model.labels))

Route encoder prepared
Known route labels: 62


In [4]:
from pyspark.sql import Row
from pyspark.ml.feature import VectorAssembler

sample_input = spark.createDataFrame([
    Row(
        published_line_name="10",
        direction_id=0,
        stop_sequence=10,
        hour=18,
        minute=30,
        day_of_week=7,
        is_weekend=1,
        is_public_holiday=0,
        journey_progress=0.50,
        previous_stop_delay=120.0,
        rolling_previous_delay=100.0,
        has_previous_delay=1
    )
])

sample_indexed = route_index_model.transform(sample_input)

tree_feature_columns = [
    "route_index",
    "direction_id",
    "stop_sequence",
    "hour",
    "minute",
    "day_of_week",
    "is_weekend",
    "is_public_holiday",
    "journey_progress",
    "previous_stop_delay",
    "rolling_previous_delay",
    "has_previous_delay"
]

assembler = VectorAssembler(
    inputCols=tree_feature_columns,
    outputCol="tree_features"
)

sample_ready = assembler.transform(sample_indexed)

sample_prediction = rf_model.transform(sample_ready)

sample_prediction.select(
    "published_line_name",
    "prediction"
).show()

+-------------------+------------------+
|published_line_name|        prediction|
+-------------------+------------------+
|                 10|104.73811674791821|
+-------------------+------------------+



In [5]:
ENCODER_PATH = r"D:\Big Data Programming Project\Final Assignment\models\route_indexer_model"

route_index_model.write() \
    .overwrite() \
    .save(ENCODER_PATH)

print("Route encoder saved to:")
print(ENCODER_PATH)

Route encoder saved to:
D:\Big Data Programming Project\Final Assignment\models\route_indexer_model


In [6]:
from pyspark.ml.regression import RandomForestRegressionModel
from pyspark.ml.feature import StringIndexerModel

MODEL_PATH = r"D:\Big Data Programming Project\Final Assignment\models\random_forest_delay_model"
ENCODER_PATH = r"D:\Big Data Programming Project\Final Assignment\models\route_indexer_model"

loaded_model = RandomForestRegressionModel.load(MODEL_PATH)
loaded_encoder = StringIndexerModel.load(ENCODER_PATH)

print("Random Forest model loaded successfully")
print("Route encoder loaded successfully")
print("Known routes:", len(loaded_encoder.labels))

Random Forest model loaded successfully
Route encoder loaded successfully
Known routes: 62
